# Signal Lab Explorer

Comprehensive exploratory notebook for browser signal-audit / signal-lab exports.

This notebook aims to expose as much structure as possible from one exported JSON session:
- metadata and capability audit
- event volume and timing
- field support / richness / constancy
- per-zone and per-kind summaries
- motion / orientation plots
- pointer / touch / wheel / scroll / drag plots
- typing / editor / selection / clipboard event plots
- correlation, missingness, and distribution views


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 300)
pd.set_option('display.max_rows', 300)
pd.set_option('display.width', 220)


## 1. Load latest JSON export

By default this notebook picks the most recent `signal_audit_*.json` file in the current folder.
Set `JSON_PATH` manually if needed.

In [ ]:
json_candidates = sorted(Path('.').glob('signal_audit_*.json'))
assert json_candidates, 'No signal_audit_*.json files found in this folder.'
JSON_PATH = max(json_candidates, key=lambda p: p.stat().st_mtime)
print('Using JSON:', JSON_PATH)

with JSON_PATH.open('r', encoding='utf-8') as f:
    payload = json.load(f)

events = pd.DataFrame(payload.get('events', []))
events.head()

## 2. Session summary


In [ ]:
capabilities = pd.Series(payload.get('capabilities', {}), name='value')
permissions = pd.Series(payload.get('permissions', {}), name='value')
environment = pd.Series(payload.get('environmentSnapshots', {}), name='value')

print('startedAtIso:', payload.get('startedAtIso'))
print('auditSessionId:', payload.get('auditSessionId'))
print('n_events:', len(events))
print('n_columns:', len(events.columns))
print('first_tRelMs:', pd.to_numeric(events.get('tRelMs'), errors='coerce').min())
print('last_tRelMs:', pd.to_numeric(events.get('tRelMs'), errors='coerce').max())

display(capabilities.to_frame('capability'))
display(permissions.to_frame('permission'))
display(environment.to_frame('environment_snapshot'))

## 3. Column audit


In [ ]:
def column_audit(df):
    rows = []
    for col in df.columns:
        s = df[col]
        non_null = s.dropna()
        numeric = pd.to_numeric(non_null, errors='coerce').dropna()
        rows.append({
            'column': col,
            'dtype': str(s.dtype),
            'non_null': int(non_null.shape[0]),
            'null': int(s.isna().sum()),
            'coverage_frac': float(non_null.shape[0] / len(df)) if len(df) else 0.0,
            'unique_non_null': int(non_null.astype(str).nunique()),
            'numeric_non_null': int(numeric.shape[0]),
            'numeric_min': float(numeric.min()) if len(numeric) else None,
            'numeric_max': float(numeric.max()) if len(numeric) else None,
            'numeric_std': float(numeric.std()) if len(numeric) > 1 else None,
            'sample_values': non_null.astype(str).unique()[:8].tolist()
        })
    return pd.DataFrame(rows).sort_values(['coverage_frac', 'unique_non_null'], ascending=[False, False])

col_audit = column_audit(events)
col_audit

In [ ]:
constant_or_near_constant = col_audit[(col_audit['non_null'] > 0) & (col_audit['unique_non_null'] <= 1)]
varying_fields = col_audit[col_audit['unique_non_null'] > 1]

print('Constant / near-constant fields:')
display(constant_or_near_constant[['column', 'non_null', 'unique_non_null', 'sample_values']])

print('Varying fields:')
display(varying_fields[['column', 'coverage_frac', 'unique_non_null', 'numeric_std']].head(100))

## 4. Event counts


In [ ]:
event_counts = events['kind'].fillna('MISSING_KIND').value_counts().rename_axis('kind').reset_index(name='count')
event_counts

In [ ]:
plt.figure(figsize=(12, 5))
event_counts.plot(kind='bar', x='kind', y='count', legend=False, ax=plt.gca())
plt.title('Event counts by kind')
plt.ylabel('Count')
plt.xticks(rotation=70, ha='right')
plt.tight_layout()
plt.show()

## 5. Per-zone event counts


In [ ]:
if 'zone' in events.columns:
    zone_counts = events.groupby(['zone', 'kind']).size().reset_index(name='count').sort_values(['zone', 'count'], ascending=[True, False])
    display(zone_counts)

    pivot = zone_counts.pivot(index='zone', columns='kind', values='count').fillna(0)
    display(pivot)
else:
    print('No zone column present.')

In [ ]:
if 'zone' in events.columns:
    pivot = events.groupby(['zone', 'kind']).size().unstack(fill_value=0)
    plt.figure(figsize=(max(10, 0.6 * pivot.shape[1]), max(4, 0.6 * pivot.shape[0])))
    plt.imshow(pivot.values, aspect='auto')
    plt.title('Zone x kind count heatmap')
    plt.xticks(range(pivot.shape[1]), pivot.columns, rotation=90)
    plt.yticks(range(pivot.shape[0]), pivot.index)
    plt.colorbar(label='Count')
    plt.tight_layout()
    plt.show()
else:
    print('No zone x kind heatmap possible.')

## 6. Event timing and density


In [ ]:
events['tRelMs_num'] = pd.to_numeric(events.get('tRelMs'), errors='coerce')

plt.figure(figsize=(11, 4))
plt.hist(events['tRelMs_num'].dropna() / 1000.0, bins=60)
plt.title('Event timing density')
plt.xlabel('Time since session start (s)')
plt.ylabel('Event count')
plt.tight_layout()
plt.show()

In [ ]:
timed = events.dropna(subset=['tRelMs_num']).copy()
timed['time_bin_s'] = (timed['tRelMs_num'] // 1000).astype(int)
events_per_second = timed.groupby('time_bin_s').size()

plt.figure(figsize=(11, 4))
plt.plot(events_per_second.index, events_per_second.values)
plt.title('Events per second')
plt.xlabel('Second from session start')
plt.ylabel('Events')
plt.tight_layout()
plt.show()

## 7. Missingness and field support heatmap


In [ ]:
support_cols = col_audit[col_audit['coverage_frac'] > 0]['column'].tolist()[:40]
support_by_kind = []
for kind, g in events.groupby('kind'):
    row = {'kind': kind}
    for col in support_cols:
        row[col] = g[col].notna().mean() if col in g.columns else 0.0
    support_by_kind.append(row)

support_df = pd.DataFrame(support_by_kind).set_index('kind').fillna(0.0)
display(support_df)

plt.figure(figsize=(max(10, 0.28 * support_df.shape[1]), max(5, 0.45 * support_df.shape[0])))
plt.imshow(support_df.values, aspect='auto', vmin=0, vmax=1)
plt.title('Field support fraction by event kind')
plt.xticks(range(support_df.shape[1]), support_df.columns, rotation=90)
plt.yticks(range(support_df.shape[0]), support_df.index)
plt.colorbar(label='Non-null fraction')
plt.tight_layout()
plt.show()

## 8. Motion signals


In [ ]:
motion = events[events['kind'] == 'devicemotion'].copy()
motion_cols = ['tRelMs_num', 'ax', 'ay', 'az', 'agx', 'agy', 'agz', 'rotAlpha', 'rotBeta', 'rotGamma', 'interval']
display(motion[motion_cols].head())

In [ ]:
if not motion.empty:
    t = motion['tRelMs_num'] / 1000.0

    plt.figure(figsize=(11, 4))
    for col in ['ax', 'ay', 'az']:
        if col in motion.columns:
            plt.plot(t, pd.to_numeric(motion[col], errors='coerce'), label=col)
    plt.title('Acceleration (without gravity)')
    plt.xlabel('Time (s)')
    plt.ylabel('Value')
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(11, 4))
    for col in ['agx', 'agy', 'agz']:
        if col in motion.columns:
            plt.plot(t, pd.to_numeric(motion[col], errors='coerce'), label=col)
    plt.title('Acceleration including gravity')
    plt.xlabel('Time (s)')
    plt.ylabel('Value')
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(11, 4))
    for col in ['rotAlpha', 'rotBeta', 'rotGamma']:
        if col in motion.columns:
            plt.plot(t, pd.to_numeric(motion[col], errors='coerce'), label=col)
    plt.title('Rotation rate')
    plt.xlabel('Time (s)')
    plt.ylabel('Value')
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print('No devicemotion events found.')

In [ ]:
if not motion.empty:
    for col in ['ax', 'ay', 'az', 'agx', 'agy', 'agz', 'rotAlpha', 'rotBeta', 'rotGamma', 'interval']:
        if col in motion.columns and pd.to_numeric(motion[col], errors='coerce').notna().any():
            plt.figure(figsize=(8, 4))
            plt.hist(pd.to_numeric(motion[col], errors='coerce').dropna(), bins=40)
            plt.title(f'Distribution of {col}')
            plt.xlabel(col)
            plt.ylabel('Count')
            plt.tight_layout()
            plt.show()

## 9. Orientation signals


In [ ]:
orientation = events[events['kind'] == 'deviceorientation'].copy()
display(orientation[['tRelMs_num', 'alpha', 'beta', 'gamma', 'absolute']].head())

In [ ]:
if not orientation.empty:
    t = orientation['tRelMs_num'] / 1000.0
    plt.figure(figsize=(11, 4))
    for col in ['alpha', 'beta', 'gamma']:
        if col in orientation.columns:
            plt.plot(t, pd.to_numeric(orientation[col], errors='coerce'), label=col)
    plt.title('Device orientation')
    plt.xlabel('Time (s)')
    plt.ylabel('Degrees')
    plt.legend()
    plt.tight_layout()
    plt.show()

    for col in ['alpha', 'beta', 'gamma']:
        if col in orientation.columns and pd.to_numeric(orientation[col], errors='coerce').notna().any():
            plt.figure(figsize=(8, 4))
            plt.hist(pd.to_numeric(orientation[col], errors='coerce').dropna(), bins=40)
            plt.title(f'Distribution of {col}')
            plt.xlabel(col)
            plt.ylabel('Count')
            plt.tight_layout()
            plt.show()
else:
    print('No deviceorientation events found.')

## 10. Pointer / touch / mouse / wheel signal overview


In [ ]:
interaction_kinds = [
    'pointerenter', 'pointerleave', 'pointerover', 'pointerout', 'pointerdown', 'pointermove', 'pointerup', 'pointercancel', 'pointerrawupdate',
    'touchstart', 'touchmove', 'touchend', 'touchcancel',
    'mousedown', 'mousemove', 'mouseup', 'mouseenter', 'mouseleave', 'mouseover', 'mouseout',
    'click', 'dblclick', 'contextmenu', 'wheel', 'scroll_zone_wheel',
    'drag_start', 'drag_move', 'drag_end',
    'gesturestart', 'gesturechange', 'gestureend'
]
interactions = events[events['kind'].isin(interaction_kinds)].copy()
display(interactions.head(20))

In [ ]:
numeric_interaction_cols = [
    'x', 'y', 'pageX', 'pageY', 'screenX', 'screenY', 'offsetX', 'offsetY', 'movementX', 'movementY',
    'pressure', 'tangentialPressure', 'width', 'height', 'tiltX', 'tiltY', 'twist', 'altitudeAngle', 'azimuthAngle',
    'force', 'radiusX', 'radiusY', 'rotationAngle', 'centroidX', 'centroidY', 'pinchDistance', 'pinchScaleApprox',
    'deltaX', 'deltaY', 'deltaZ', 'relativeX', 'relativeY', 'dx', 'dy', 'speedPxPerMs', 'scale', 'rotation'
]

for col in numeric_interaction_cols:
    if col in interactions.columns and pd.to_numeric(interactions[col], errors='coerce').notna().any():
        plt.figure(figsize=(8, 4))
        plt.hist(pd.to_numeric(interactions[col], errors='coerce').dropna(), bins=40)
        plt.title(f'Distribution of {col}')
        plt.xlabel(col)
        plt.ylabel('Count')
        plt.tight_layout()
        plt.show()

In [ ]:
if {'x', 'y'}.issubset(interactions.columns):
    path_df = interactions.dropna(subset=['x', 'y']).copy()
    if not path_df.empty:
        plt.figure(figsize=(7, 7))
        plt.scatter(pd.to_numeric(path_df['x'], errors='coerce'), pd.to_numeric(path_df['y'], errors='coerce'), s=6, alpha=0.5)
        plt.title('All interaction points')
        plt.xlabel('x')
        plt.ylabel('y')
        plt.gca().invert_yaxis()
        plt.tight_layout()
        plt.show()
else:
    print('No x/y path plot possible.')

In [ ]:
if {'x', 'y', 'kind'}.issubset(interactions.columns):
    for kind in interactions['kind'].dropna().unique():
        g = interactions[interactions['kind'] == kind].dropna(subset=['x', 'y'])
        if g.empty:
            continue
        plt.figure(figsize=(6, 6))
        plt.scatter(pd.to_numeric(g['x'], errors='coerce'), pd.to_numeric(g['y'], errors='coerce'), s=6, alpha=0.5)
        plt.title(f'Interaction points: {kind}')
        plt.xlabel('x')
        plt.ylabel('y')
        plt.gca().invert_yaxis()
        plt.tight_layout()
        plt.show()

## 11. Pointer / touch time series


In [ ]:
time_cols = ['x', 'y', 'pressure', 'width', 'height', 'force', 'radiusX', 'radiusY', 'rotationAngle', 'pinchDistance', 'deltaY', 'relativeX', 'relativeY', 'speedPxPerMs']
for kind in ['pointermove', 'touchmove', 'wheel', 'drag_move', 'gesturechange']:
    g = interactions[interactions['kind'] == kind].copy()
    if g.empty:
        continue
    t = pd.to_numeric(g['tRelMs_num'], errors='coerce') / 1000.0
    for col in time_cols:
        if col in g.columns and pd.to_numeric(g[col], errors='coerce').notna().any():
            plt.figure(figsize=(11, 4))
            plt.plot(t, pd.to_numeric(g[col], errors='coerce'))
            plt.title(f'{kind}: {col} over time')
            plt.xlabel('Time (s)')
            plt.ylabel(col)
            plt.tight_layout()
            plt.show()

## 12. Scroll behaviour


In [ ]:
scroll_df = events[events['kind'] == 'scroll'].copy()
display(scroll_df[['tRelMs_num', 'scrollTop', 'scrollLeft', 'scrollHeight', 'clientHeight', 'zone']].head())

In [ ]:
if not scroll_df.empty and 'scrollTop' in scroll_df.columns:
    plt.figure(figsize=(11, 4))
    plt.plot(scroll_df['tRelMs_num'] / 1000.0, pd.to_numeric(scroll_df['scrollTop'], errors='coerce'))
    plt.title('scrollTop over time')
    plt.xlabel('Time (s)')
    plt.ylabel('scrollTop')
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(8, 4))
    plt.hist(pd.to_numeric(scroll_df['scrollTop'], errors='coerce').dropna(), bins=40)
    plt.title('Distribution of scrollTop')
    plt.xlabel('scrollTop')
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()
else:
    print('No scroll plot possible.')

## 13. Drag behaviour


In [ ]:
drag_df = events[events['kind'].isin(['drag_start', 'drag_move', 'drag_end'])].copy()
display(drag_df.head(20))

In [ ]:
if not drag_df.empty and {'relativeX', 'relativeY'}.issubset(drag_df.columns):
    dm = drag_df[drag_df['kind'] == 'drag_move'].dropna(subset=['relativeX', 'relativeY'])
    if not dm.empty:
        plt.figure(figsize=(7, 7))
        plt.plot(pd.to_numeric(dm['relativeX'], errors='coerce'), pd.to_numeric(dm['relativeY'], errors='coerce'), marker='o', markersize=2)
        plt.title('Drag path (relative coordinates)')
        plt.xlabel('relativeX')
        plt.ylabel('relativeY')
        plt.gca().invert_yaxis()
        plt.tight_layout()
        plt.show()

    for col in ['relativeX', 'relativeY', 'dx', 'dy', 'speedPxPerMs']:
        if col in dm.columns and pd.to_numeric(dm[col], errors='coerce').notna().any():
            plt.figure(figsize=(10, 4))
            plt.plot(dm['tRelMs_num'] / 1000.0, pd.to_numeric(dm[col], errors='coerce'))
            plt.title(f'Drag move: {col} over time')
            plt.xlabel('Time (s)')
            plt.ylabel(col)
            plt.tight_layout()
            plt.show()
else:
    print('No drag plots possible.')

## 14. Typing / editor / clipboard / selection


In [ ]:
text_like_kinds = [
    'keydown', 'keyup', 'beforeinput', 'input', 'select', 'focus', 'blur',
    'compositionstart', 'compositionupdate', 'compositionend',
    'copy', 'cut', 'paste',
    'editor_focus', 'editor_blur', 'editor_beforeinput', 'editor_input', 'editor_keydown', 'editor_keyup',
    'editor_compositionstart', 'editor_compositionupdate', 'editor_compositionend',
    'editor_copy', 'editor_cut', 'editor_paste',
    'selectionchange'
]
text_df = events[events['kind'].isin(text_like_kinds)].copy()
display(text_df.head(40))

In [ ]:
if not text_df.empty:
    text_counts = text_df['kind'].value_counts().sort_values(ascending=False)
    plt.figure(figsize=(11, 4))
    text_counts.plot(kind='bar')
    plt.title('Text/editor-related event counts')
    plt.ylabel('Count')
    plt.xticks(rotation=70, ha='right')
    plt.tight_layout()
    plt.show()

    for col in ['valueLength', 'lineCount', 'selectionStart', 'selectionEnd', 'textLength', 'childNodeCount', 'clipboardTextLength', 'dataLength']:
        if col in text_df.columns and pd.to_numeric(text_df[col], errors='coerce').notna().any():
            plt.figure(figsize=(10, 4))
            plt.plot(text_df['tRelMs_num'] / 1000.0, pd.to_numeric(text_df[col], errors='coerce'), marker='o', linewidth=1)
            plt.title(f'{col} over time')
            plt.xlabel('Time (s)')
            plt.ylabel(col)
            plt.tight_layout()
            plt.show()

            plt.figure(figsize=(8, 4))
            plt.hist(pd.to_numeric(text_df[col], errors='coerce').dropna(), bins=30)
            plt.title(f'Distribution of {col}')
            plt.xlabel(col)
            plt.ylabel('Count')
            plt.tight_layout()
            plt.show()
else:
    print('No text/editor-related events found.')

## 15. Lifecycle / environment events


In [ ]:
lifecycle_kinds = [
    'session_start', 'audit_init', 'window_focus', 'window_blur', 'pageshow', 'pagehide',
    'resize', 'orientationchange', 'screen_orientation_change', 'visibilitychange',
    'online', 'offline', 'connection_change', 'visual_viewport_resize', 'visual_viewport_scroll',
    'battery_snapshot', 'motion_permission_granted', 'permission_change', 'geolocation', 'geolocation_error',
    'generic_accelerometer', 'generic_linear_acceleration', 'generic_gyroscope', 'generic_magnetometer',
    'generic_absolute_orientation', 'generic_relative_orientation', 'generic_ambient_light',
    'generic_sensor_error', 'generic_sensor_start_error', 'vibration_test', 'vibration_unsupported'
]
lifecycle_df = events[events['kind'].isin(lifecycle_kinds)].copy()
display(lifecycle_df.head(50))

In [ ]:
if not lifecycle_df.empty:
    life_counts = lifecycle_df['kind'].value_counts().sort_values(ascending=False)
    plt.figure(figsize=(11, 4))
    life_counts.plot(kind='bar')
    plt.title('Lifecycle / environment event counts')
    plt.ylabel('Count')
    plt.xticks(rotation=70, ha='right')
    plt.tight_layout()
    plt.show()

    for col in ['viewportWidth', 'viewportHeight', 'offsetLeft', 'offsetTop', 'pageLeft', 'pageTop', 'scale', 'latitude', 'longitude', 'accuracy', 'illuminance', 'x', 'y', 'z']:
        if col in lifecycle_df.columns and pd.to_numeric(lifecycle_df[col], errors='coerce').notna().any():
            plt.figure(figsize=(10, 4))
            plt.plot(lifecycle_df['tRelMs_num'] / 1000.0, pd.to_numeric(lifecycle_df[col], errors='coerce'), marker='o', linewidth=1)
            plt.title(f'{col} over time (lifecycle/environment events)')
            plt.xlabel('Time (s)')
            plt.ylabel(col)
            plt.tight_layout()
            plt.show()
else:
    print('No lifecycle/environment events found.')

## 16. Categorical fields


In [ ]:
categorical_candidates = ['zone', 'pointerType', 'keyClass', 'code', 'inputType', 'selectionDirection', 'selectionType', 'visibilityState', 'sensorType']
for col in categorical_candidates:
    if col in events.columns and events[col].notna().any():
        vc = events[col].astype(str).value_counts().head(30)
        display(vc.to_frame(name='count'))
        plt.figure(figsize=(10, 4))
        vc.plot(kind='bar')
        plt.title(f'Top values for {col}')
        plt.ylabel('Count')
        plt.xticks(rotation=70, ha='right')
        plt.tight_layout()
        plt.show()

## 17. Numeric correlation matrix


In [ ]:
numeric_df = events.copy()
for col in numeric_df.columns:
    numeric_df[col] = pd.to_numeric(numeric_df[col], errors='ignore')

numeric_cols = [c for c in numeric_df.columns if pd.api.types.is_numeric_dtype(numeric_df[c])]
numeric_cols = [c for c in numeric_cols if numeric_df[c].notna().sum() >= 10]
corr_cols = numeric_cols[:30]

if corr_cols:
    corr = numeric_df[corr_cols].corr(numeric_only=True)
    display(corr)

    plt.figure(figsize=(max(8, 0.45 * len(corr_cols)), max(6, 0.45 * len(corr_cols))))
    plt.imshow(corr.values, vmin=-1, vmax=1)
    plt.title('Numeric correlation matrix (first 30 numeric columns)')
    plt.xticks(range(len(corr_cols)), corr_cols, rotation=90)
    plt.yticks(range(len(corr_cols)), corr_cols)
    plt.colorbar(label='Correlation')
    plt.tight_layout()
    plt.show()
else:
    print('No numeric correlation matrix possible.')

## 18. Automated plot sweep for numeric columns

This section tries to plot every sufficiently supported numeric column once as a time series and once as a histogram.
It is intentionally broad so you can see as much of the signal surface as possible.

In [ ]:
sweep_candidates = []
for col in events.columns:
    if col in ['tRelMs', 'tRelMs_num']:
        continue
    numeric = pd.to_numeric(events[col], errors='coerce') if col in events.columns else pd.Series(dtype=float)
    if numeric.notna().sum() >= 10 and numeric.nunique(dropna=True) > 1:
        sweep_candidates.append(col)

print('Sweep candidates:', len(sweep_candidates))
print(sweep_candidates)

In [ ]:
for col in sweep_candidates:
    vals = pd.to_numeric(events[col], errors='coerce')

    plt.figure(figsize=(10, 4))
    plt.plot(events['tRelMs_num'] / 1000.0, vals, marker='o', linewidth=1, markersize=2)
    plt.title(f'{col} over time')
    plt.xlabel('Time (s)')
    plt.ylabel(col)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(8, 4))
    plt.hist(vals.dropna(), bins=40)
    plt.title(f'Distribution of {col}')
    plt.xlabel(col)
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()

## 19. Quick interpretation helper


In [ ]:
def interpret_field(row):
    if row['non_null'] == 0:
        return 'all null'
    if row['unique_non_null'] <= 1:
        return 'constant or near-constant'
    if row['coverage_frac'] < 0.01:
        return 'very sparse but varying'
    if row['coverage_frac'] < 0.10:
        return 'sparse but varying'
    if row['numeric_std'] is not None and row['numeric_std'] == 0:
        return 'constant numeric'
    return 'varying'

quick_interpretation = col_audit[['column', 'coverage_frac', 'unique_non_null', 'numeric_std', 'sample_values']].copy()
quick_interpretation['interpretation'] = col_audit.apply(interpret_field, axis=1)
quick_interpretation.sort_values(['interpretation', 'coverage_frac'], ascending=[True, False])